# Task 2 — Pose estimation models benchmark (speed + accuracy proxy)

This notebook runs **two pose estimators** over your *real* squat videos in `videos_squat/`:

1. **YOLO pose** (Ultralytics `yolo11n-pose.pt`) → 17 COCO keypoints
2. **OpenPose-style** (OpenCV DNN using the Caffe OpenPose COCO18 model) → mapped to 17 COCO keypoints

We compare models on:
- **Speed**: estimated frames/second during extraction.
- **Accuracy proxy**: how well extracted **knee angle** separates frame-level knee-error intervals (`error_knees_forward` + `error_knees_inward`).

> Why not MediaPipe here? In this container, MediaPipe “tasks” fails to initialize due to a missing `libEGL.so.1` dependency. YOLO + OpenPose-DNN are runnable without that system library.


In [ ]:
from pathlib import Path
import os

from src.dataset_labels import load_dataset_manifest
from src.pose_estimators import YoloPose17, OpenPoseDnn17
from src.benchmark_pose_estimators import benchmark_pose_estimator_on_dataset

root = Path.cwd()
manifest = load_dataset_manifest(root_dir=root, videos_dir_name="videos_squat")

# Use val split for quick benchmarking
val_dataset = [m for m in manifest if m.split == "val"]
val_dataset = sorted(val_dataset, key=lambda m: m.video_id)
print("Val videos:", len(val_dataset))
print("First few:", [m.video_id for m in val_dataset[:5]])


In [ ]:
# ---- Download OpenPose Caffe model files (if needed) ----
import urllib.request

models_dir = root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Prototxt: from OpenCV extra testdata
pose_proto = models_dir / "openpose_pose_mpi_faster_4_stages.prototxt"
pose_proto_url = "https://raw.githubusercontent.com/opencv/opencv_extra/4.x/testdata/dnn/openpose_pose_mpi_faster_4_stages.prototxt"

# Caffemodel: from CMU OpenPose
pose_weights = models_dir / "pose_iter_160000.caffemodel"
pose_weights_url = "https://huggingface.co/camenduru/openpose/resolve/f4a22b0e6fa2a4a2b1e2d50bd589e8bb11ebea7c/pose_iter_160000.caffemodel"

def maybe_download(path: Path, url: str):
    if path.exists() and path.stat().st_size > 0:
        print(f"Using existing: {path}")
        return
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, path)

maybe_download(pose_proto, pose_proto_url)
maybe_download(pose_weights, pose_weights_url)

print("OpenPose model files ready")


In [ ]:
# ---- Instantiate estimators ----
yolo = YoloPose17(
    model_name_or_path="yolo11n-pose.pt",
    conf_threshold=0.08,
    max_det=1,
)

openpose = OpenPoseDnn17(
    prototxt_path=str(pose_proto),
    caffemodel_path=str(pose_weights),
    conf_threshold=0.05,
)

print("Estimators ready")


In [ ]:
import pandas as pd

# Quick benchmark settings (kept small for fast notebook execution)
benchmark_kwargs = dict(
    max_videos=3,
    max_frames_per_video=140,
    frame_sample_stride=3,
    use_forward=True,
    use_inward=True,
    verbose=True,
)

metrics_yolo = benchmark_pose_estimator_on_dataset(
    model_name="yolo11n-pose",
    estimator=yolo,
    dataset=val_dataset,
    **benchmark_kwargs,
)

metrics_openpose = benchmark_pose_estimator_on_dataset(
    model_name="openpose-dnn",
    estimator=openpose,
    dataset=val_dataset,
    **benchmark_kwargs,
)

df = pd.DataFrame([
    metrics_yolo.__dict__,
    metrics_openpose.__dict__,
])
df


In [ ]:
# ---- Plot quick comparison ----
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.bar(df["model_name"], df["fps_processing"])
plt.title("Pose estimator speed (frames/sec)")
plt.ylabel("frames/sec")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

speed_plot_path = root / "models" / "pose_benchmark_speed.png"
plt.savefig(speed_plot_path, dpi=160)
speed_plot_path


## Selecting the best model (for your downstream pipeline)

Use the benchmark output to choose the pose estimator:
- Prefer the one with **higher ROC-AUC / F1** on the knee-error proxy.
- If speed is critical for live overlay, prefer the higher **frames/sec**.

This notebook compares two realistic options you can run in your environment. If later you can enable MediaPipe (fixing the EGL dependency), you can extend the benchmark by adding a third model.
